# Using Gradio and the `Meta-Llama-3.1-8B-Instruct` LLM to create Meeting Minutes from an Audio File

Ken Wood

4/1/2025

The goal of this product is to input an audio transcription of a meeting and use Meta's Llama LLM to product a set of organized meeting minutes, including action items.

The audio file we'll use for this notebook are some Denver City Council meeting minutes [link](https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing) and selected a portion of the meeting to transcribe.

The original HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).


#### Install and import necessary packages

In [1]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio openai httpx==0.27.2

In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gradio as gr

#### Mount Google Drive so Colab can see the audio file.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Establish some LLM parameters

In [5]:
# Constants

AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Sign in to OpenAI using Secrets in Colab

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [ ]:
audio_file_path = #/content/drive/MyDrive/Colab_Notebooks/LLMs_Course_Udemy/data files/denver_extract.mp3"

#### Download denver_extract.mp3

https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


####  Create the meeting minutes function


In [6]:
def create_meeting_minutes(audio_file_path):

  # Use the Whisper OpenAI model to convert the Audio to Text

  audio_file = open(audio_file_path, "rb")
  transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")

  # Create system_message and user_prompt (which includes the input audio file transcription) for the Llama LLM so that it understands its role.

  system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
  user_prompt = f"Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcription}"

  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
    ]

  # Establish a quantization configuration of 4 bits to reduce memory size
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )

  # Tokenize the audio file transcription text and stream it to the LLM
  tokenizer = AutoTokenizer.from_pretrained(LLAMA)
  tokenizer.pad_token = tokenizer.eos_token
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
  streamer = TextStreamer(tokenizer)

  # LLM Model
  model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

  # Generate outputs
  outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
  response = tokenizer.decode(outputs[0])

  return display(Markdown(response))

#### Initiate the Gradio interface

In [ ]:
view = gr.Interface(
    fn=create_meeting_minutes,
    inputs=[gr.Textbox(label="Audio File Path:", lines=6)],
    outputs=[gr.Markdown(label="Meeting Minutes")],
    flagging_mode="never"
)
view.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fb3267718c3df31421.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.
and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings b

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.
and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism and you'll hear a little bit more about our confluence week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous People's Day. So thank you. Thank you so much and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the United States of America, and to the republic for which it stands, one nation, under God, indivisible, with liberty and justice for all. All right. Thank you, Councilman Lopez. Madam Secretary, roll call. Clerk. Here. Espinosa. Here. Flynn. Here. Gilmour. Here. Here. Cashman. Here. Kenneche. Here. Lopez. Here. New. Here. Ortega. Here. Sussman. Here. Mr. President. Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Seeing none. Minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of Council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family-friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids and there are tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do we have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening. Proclamation 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President, with pride. Proclamation number 17, well, let me just say this differently. Proclamation number 1127, series of 2017, an observance of the second annual Indigenous Peoples Day in the City and County of Denver. Whereas the Council of the City and County of Denver recognizes that the Indigenous Peoples have lived and flourished on the lands known as the Americas since time immemorial and that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes, which include the Southern Ute, the Ute Mountain, Ute tribes of Colorado. And whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes and the City and County of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3, 2016, the City and County of Denver unanimously passed Council Bill 801, series of 2016, officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado. And whereas the Council of the City and County of Denver continues to recognize and value the vast contributions made to the community through Indigenous Peoples' knowledge, science, philosophy, arts, and culture. And through these contributions, the City of Denver has developed and thrived. Whereas the Indigenous community, especially youth, have made great efforts this year to draw attention to the contributions of Indigenous people, including Confluence Week, drawing record attendance to a National Indigenous Youth Leadership Conference, leading conversations on inclusion with their peers, and supporting increased Indigenous youth participation in science and engineering. Now, therefore, be it proclaimed by the Council of the City and County of Denver, Section 1, that the Council of the City and County of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to our history, our past, our present, and future, and continues to promote the education of the Denver community about these historical and contemporary contributions of Indigenous people. Section 2, that the City and County of Denver, Colorado, does hereby observe October 9, 2017 as Indigenous Peoples Day. Section 3, that the Clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation, and that a copy be transmitted to the Denver American Indian Commission, the City and County of Denver School District No. 1, and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt. Mr. President, I move that Proclamation No. 1127, Series of 2017, be adopted. All right. It has been moved and seconded. It comes from the members of Council. Councilman Lopez. Thank you, Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for the third time, but as Indigenous Peoples Day in Denver, officially for the second time. It's always awesome to be able to see not just this proclamation come by my desk, but to see so many different people from our community in our Council Chambers. It was a very beautiful piece of artwork that you presented to us earlier, and it is exactly the spirit that we drafted this proclamation and this actual, the ordinance that created Indigenous Peoples Day. And we sat down and wrote it, and as a community, we couldn't think of anything else to begin except for the confluence of the two rivers. And those confluence of the two rivers created such a great city. And we live in such an amazing city, and we're all proud of it, and sometimes we, and a lot of people from all over the country or all over the world are proud of it, and sometimes a little too proud of it is to tell them to go back home. But I'm kidding when I say that. But the really nice thing about this is that we are celebrating Indigenous Peoples Day out of pride for who we are, who we are as a city, and the contributions of Indigenous Peoples to the city, not out of spite, not out of a replacement of one culture over the other, or out of contempt or disrespect. I think of a quote that Cesar Chavez made very popular, and it stuck with me for a very long time, and any time I have the opportunity to speak in front of children, and especially children in our community, that they often second-guess themselves on where they're coming from, who they are. And I always say that it's very important to be proud of where you're from, and the quote that I use from Cesar Chavez is, you know, pride in one's own culture does not require contempt or disrespect of another. Right? And that's very important. It's very important for us to recognize that no matter who we are, where we come from in this society, your pride in your own culture does not require the contempt or disrespect of another. And man, what a year for that to just sit on our shoulders for a while, for us to think about. Right? And so I wanted to just thank you all. I think the commission, there's going to be a couple individuals that are going to come and speak. Thank you for your art, your lovely artwork, for us to see what's in your heart and what now has become, probably is going to be a very important symbol for the community. And also just for the work, the daily work, every single day. We still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners. Right? In poverty. Without access to services. Right? Without access to sobriety or even housing or jobs. And what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores. Right? So we have a lot of work to do. And these kind of proclamations and this day is not a day off, it's a day on in Denver. Right? And addressing those critical issues. So I know that my colleagues are very supportive. I'm going to ask you to support this proclamation. As I know you always have done in the past. I'm very proud of today. Oh, and we made Time Magazine and Newsweek once again today. As being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you, Councilman Lopez, and thank you for sponsoring this. Councilman Martega? Mr. President, I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us. I want to thank him for bringing this forward. And really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's Office when the Commission on Indian Affairs was and had the benefit of being able to go down to the Four Corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about 100 years. And just being able to participate in that powwow was pretty awesome. And for those of you who continue to participate in the annual powwow, it's such a great opportunity for everybody else to enjoy so many of the contributions of the culture. I mean, to see that the dance continues to be carried on as well as the Native language from generation to generation is just so incredible because in so many cultures, people have come here and assimilated to the norms here and they lose their language and lose a lot of the culture. And in the Native community, that hasn't happened. Commitment to just passing that on from generation to generation is so important. So I'm happy to be a co-sponsor of this tonight. Thank you. All right. Thank you, Councilwoman Ortega. Councilwoman Caniche. Thank you very much. And I also want to thank my colleague for bringing this forward. And I just wanted to say a word to the artist about how beautiful and moving I thought this logo was and your description of it. And I think one of the things that is clear is the words sometimes don't convey the power of imagery or music or the other pieces that make up culture. And so I think the art is so important. And when you talked about water, I was also thinking about land. And I guess I just wanted to say thank you. Many of the Native American peoples of Colorado have been at the forefront, or actually nationally, of defending some of the public lands that have been protected over the last few years that are under attack right now. And there are places that the communities have fought to protect but that everyone gets to enjoy. And so I just think that it's an example of where cultural preservation intersects with environmental protection, with recreation and all of the other ways that public lands are so important. And so I think I just wanted to say thank you for that because I think we have some very sacred places in our country that are at risk right now. And so as we celebrate, I appreciate that there's still a piece of resistance in here. And I think that I just want to mention a solidarity. I mentioned a feeling of solidarity with that resistance. So thank you, and happy Confluence Week. Thank you, Councilwoman Knitsch. And seeing no other comments, I'll just say a couple. And in a time of such divisive ugliness and just despicable behavior from our leadership, the reason I'm so supportive of Indigenous Peoples' Day is because it means inclusivity. It means respecting all, respecting those who have been silenced on purpose for a long time and whose history has not been told. And so we celebrate inclusivity in the face of such evil times, honestly.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Meeting Minutes**
===============

**Summary**
----------

*   Date: October 9, 2017
*   Location: Denver City Council Meeting
*   Attendees: Council Members Lopez, Clark, Martega, Caniche, Ortega, Flynn, Gilmour, Cashman, Espinosa, Kenneche, New, Sussman, Mr. President
*   Purpose: To discuss and adopt Proclamation No. 1127, Series of 2017, an observance of the second annual Indigenous Peoples Day in the City and County of Denver.

**Discussion Points**
--------------------

*   Councilman Lopez presented the proclamation, highlighting the importance of Indigenous Peoples Day and the contributions of Indigenous people to the city's history, present, and future.
*   Councilman Lopez emphasized the significance of pride in one's own culture without requiring contempt or disrespect of another culture.
*   Councilwoman Ortega shared her experience with the Native American community and the importance of preserving their culture and traditions.
*   Councilwoman Caniche spoke about the intersection of cultural preservation and environmental protection, highlighting the importance of public lands and the role of Native American communities in defending them.

**Takeaways**
------------

*   The city of Denver celebrates and honors the cultural and foundational contributions of Indigenous people to its history, present, and future.
*   The proclamation recognizes the importance of inclusivity and respecting all, especially those who have been silenced or marginalized.
*   The city continues to promote the education of the Denver community about the historical and contemporary contributions of Indigenous people.

**Action Items**
---------------

*   Adopt Proclamation No. 1127, Series of 2017, an observance of the second annual Indigenous Peoples Day in the City and County of Denver.
*   The Clerk of the City and County of Denver shall attest and affix the seal of the City and County of Denver to this proclamation.
*   A copy of the proclamation shall be transmitted to the Denver American Indian Commission, the City and County of Denver School District No. 1, and the Colorado Commission on Indian Affairs.

**Owners**
----------

*   Councilman Lopez: Sponsor of the proclamation
*   Councilwoman Ortega: Co-sponsor of the proclamation
*   Councilwoman Caniche: Provided comments and expressed solidarity with the resistance to protect public lands and cultural preservation.<|eot_id|>